In [1]:
import pandas as pd
import numpy as np

print("Data Preparation Started 🚀")

df = pd.read_csv(
    r"C:\Users\lenovo\OneDrive\Desktop\ai-ml project\fraud-detection-mlops\data\raw\credit_card_fraud_10k.csv"
)

print("Shape:", df.shape)
df.head()

Data Preparation Started 🚀
Shape: (10000, 10)


,transaction_id,amount,transaction_hour,merchant_category,foreign_transaction,location_mismatch,device_trust_score,velocity_last_24h,cardholder_age,is_fraud
0,1,84.47,22,Electronics,0,0,66,3,40,0
1,2,541.82,3,Travel,1,0,87,1,64,0
2,3,237.01,17,Grocery,0,0,49,1,61,0
3,4,164.33,4,Grocery,0,1,72,3,34,0
4,5,30.53,15,Food,0,0,79,0,44,0


In [2]:
X = df.drop(columns=["is_fraud", "transaction_id"])
y = df["is_fraud"]

print("Features:")
print(X.columns.tolist())

print("\nTarget:")
print(y.name)

print("\nX shape:", X.shape)
print("y shape:", y.shape)

Features:
['amount', 'transaction_hour', 'merchant_category', 'foreign_transaction', 'location_mismatch', 'device_trust_score', 'velocity_last_24h', 'cardholder_age']

Target:
is_fraud

X shape: (10000, 8)
y shape: (10000,)


In [3]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts(normalize=True) * 100)

print("\nTesting target distribution:")
print(y_test.value_counts(normalize=True) * 100)

Training data: (8000, 8)
Testing data: (2000, 8)

Training target distribution:
is_fraud
0    98.4875
1     1.5125
Name: proportion, dtype: float64

Testing target distribution:
is_fraud
0    98.5
1     1.5
Name: proportion, dtype: float64


In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

In [5]:
categorical_cols = ["merchant_category"]

numerical_cols = [
    "amount",
    "transaction_hour",
    "foreign_transaction",
    "location_mismatch",
    "device_trust_score",
    "velocity_last_24h",
    "cardholder_age"
]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
    ]
)

print("Preprocessor created successfully ✅")

Preprocessor created successfully ✅


In [6]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Training data shape:", X_train_processed.shape)
print("Testing data shape:", X_test_processed.shape)

Training data shape: (8000, 12)
Testing data shape: (2000, 12)


In [7]:
from sklearn.linear_model import LogisticRegression

# Baseline model
log_reg = LogisticRegression(
    class_weight="balanced",
    random_state=42,
    max_iter=1000
)

# Train
log_reg.fit(X_train_processed, y_train)

print("Logistic Regression trained successfully ✅")

Logistic Regression trained successfully ✅


In [8]:
y_pred = log_reg.predict(X_test_processed)
y_prob = log_reg.predict_proba(X_test_processed)[:, 1]

print("Predictions generated successfully ✅")
print("Predictions:", y_pred[:10])
print("Fraud probabilities:", y_prob[:10])

Predictions generated successfully ✅
Predictions: [0 0 0 0 0 0 0 0 0 0]
Fraud probabilities: [1.16608676e-03 5.35650014e-04 1.72252501e-06 5.25807637e-03
 5.92792730e-05 2.80740559e-02 2.61911384e-10 1.25852710e-02
 6.25990598e-07 1.06956599e-03]


In [10]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    average_precision_score,
    roc_auc_score
)

print("Classification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nROC-AUC:", roc_auc_score(y_test, y_prob))
print("PR-AUC:", average_precision_score(y_test, y_prob))

Classification Report:


              precision    recall  f1-score   support

           0       1.00      0.96      0.98      1970
           1       0.26      0.97      0.41        30

    accuracy                           0.96      2000
   macro avg       0.63      0.96      0.69      2000
weighted avg       0.99      0.96      0.97      2000


Confusion Matrix:
[[1887   83]
 [   1   29]]

ROC-AUC: 0.9932825719120135
PR-AUC: 0.7364463488041411


In [11]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train_processed, y_train)

print("Random Forest trained successfully ✅")

Random Forest trained successfully ✅


In [12]:
rf_pred = rf_model.predict(X_test_processed)
rf_prob = rf_model.predict_proba(X_test_processed)[:, 1]

print("Random Forest predictions generated successfully ✅")

Random Forest predictions generated successfully ✅


In [13]:
print("Random Forest Classification Report:")
print(classification_report(y_test, rf_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, rf_pred))

print("\nROC-AUC:", roc_auc_score(y_test, rf_prob))
print("PR-AUC:", average_precision_score(y_test, rf_prob))

Random Forest Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1970
           1       1.00      0.83      0.91        30

    accuracy                           1.00      2000
   macro avg       1.00      0.92      0.95      2000
weighted avg       1.00      1.00      1.00      2000


Confusion Matrix:
[[1970    0]
 [   5   25]]

ROC-AUC: 1.0
PR-AUC: 0.9999999999999999


In [14]:
feature_importance = pd.Series(
    rf_model.feature_importances_,
    index=preprocessor.get_feature_names_out()
)

feature_importance = feature_importance.sort_values(ascending=False)

feature_importance.head(15)

num__transaction_hour                 0.260040
num__device_trust_score               0.244203
num__foreign_transaction              0.141941
num__velocity_last_24h                0.137851
num__location_mismatch                0.118193
num__amount                           0.047686
num__cardholder_age                   0.028763
cat__merchant_category_Clothing       0.004919
cat__merchant_category_Travel         0.004745
cat__merchant_category_Electronics    0.004588
cat__merchant_category_Grocery        0.004342
cat__merchant_category_Food           0.002731
dtype: float64

In [16]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier

param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [10, 20],
    "min_samples_split": [2, 5]
}

grid_search = GridSearchCV(
    estimator=RandomForestClassifier(
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ),
    param_grid=param_grid,
    scoring="average_precision",
    cv=3,
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train_processed, y_train)

print("Best Parameters:", grid_search.best_params_)
print("Best CV PR-AUC:", grid_search.best_score_)

Fitting 3 folds for each of 8 candidates, totalling 24 fits


KeyboardInterrupt: 

In [17]:
print("Using baseline Random Forest model for production pipeline.")
print("Model:", type(rf_model).__name__)
print("Number of trees:", rf_model.n_estimators)

Using baseline Random Forest model for production pipeline.
Model: RandomForestClassifier
Number of trees: 300


In [18]:
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score

thresholds = [0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90]

threshold_results = []

for threshold in thresholds:
    predictions = (rf_prob >= threshold).astype(int)

    precision = precision_score(y_test, predictions, zero_division=0)
    recall = recall_score(y_test, predictions, zero_division=0)
    f1 = f1_score(y_test, predictions, zero_division=0)

    threshold_results.append({
        "threshold": threshold,
        "precision": precision,
        "recall": recall,
        "f1_score": f1
    })

threshold_df = pd.DataFrame(threshold_results)

threshold_df

,threshold,precision,recall,f1_score
0,0.1,0.344828,1.000000,0.512821
1,0.2,0.697674,1.000000,0.821918
2,0.3,1.000000,1.000000,1.000000
3,0.4,1.000000,0.966667,0.983051
4,0.5,1.000000,0.833333,0.909091
5,0.6,1.000000,0.633333,0.775510
6,0.7,1.000000,0.433333,0.604651
7,0.8,1.000000,0.266667,0.421053
8,0.9,1.000000,0.166667,0.285714


In [19]:
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import precision_recall_curve

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

rf_cv = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

cv_prob = cross_val_predict(
    rf_cv,
    X_train_processed,
    y_train,
    cv=cv,
    method="predict_proba",
    n_jobs=1
)[:, 1]

print("Cross-validation predictions generated successfully ✅")

Cross-validation predictions generated successfully ✅


In [20]:
thresholds = np.arange(0.10, 0.91, 0.05)

cv_threshold_results = []

for threshold in thresholds:
    cv_predictions = (cv_prob >= threshold).astype(int)

    precision = precision_score(
        y_train,
        cv_predictions,
        zero_division=0
    )

    recall = recall_score(
        y_train,
        cv_predictions,
        zero_division=0
    )

    f1 = f1_score(
        y_train,
        cv_predictions,
        zero_division=0
    )

    cv_threshold_results.append({
        "threshold": round(threshold, 2),
        "precision": precision,
        "recall": recall,
        "f1_score": f1
    })

cv_threshold_df = pd.DataFrame(cv_threshold_results)

cv_threshold_df.sort_values(
    "f1_score",
    ascending=False
).head(10)

,threshold,precision,recall,f1_score
4,0.30,0.934426,0.942149,0.938272
3,0.25,0.874074,0.975207,0.921875
5,0.35,0.971963,0.859504,0.912281
6,0.40,0.989796,0.801653,0.885845
7,0.45,1.000000,0.735537,0.847619
2,0.20,0.694118,0.975207,0.810997
8,0.50,1.000000,0.669421,0.801980
9,0.55,1.000000,0.611570,0.758974
10,0.60,1.000000,0.537190,0.698925
1,0.15,0.500000,0.991736,0.664820


In [1]:
import mlflow

print("MLflow imported successfully ✅")
print("Version:", mlflow.__version__)


MLflow imported successfully ✅
Version: 3.16.1


In [2]:
mlflow.set_tracking_uri("file:./mlruns")

print("Tracking URI configured successfully ✅")
print("Tracking URI:", mlflow.get_tracking_uri())


Tracking URI configured successfully ✅
Tracking URI: file:./mlruns


In [3]:
mlflow.set_experiment("Fraud Detection - Random Forest")

print("MLflow experiment configured successfully ✅")

MlflowException: The filesystem tracking backend (e.g., './mlruns') is in maintenance mode and will not receive further updates. Please migrate to a database backend (e.g., 'sqlite:///mlflow.db') to access the latest MLflow features. The `mlflow migrate-filestore` tool migrates your existing data losslessly. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance. If the filesystem backend is required for your workflow, set `MLFLOW_ALLOW_FILE_STORE=true` to opt out of this exception.

In [4]:
import mlflow

mlflow.set_tracking_uri("sqlite:///mlflow.db")

print("MLflow database tracking configured successfully ✅")
print("Tracking URI:", mlflow.get_tracking_uri())

MLflow database tracking configured successfully ✅
Tracking URI: sqlite:///mlflow.db


In [5]:
mlflow.set_experiment("Fraud Detection - Random Forest")

print("MLflow experiment configured successfully ✅")

2026/09/29 13:53:27 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/09/29 13:53:27 INFO mlflow.store.db.utils: Updating database tables
2026/09/29 13:53:29 INFO mlflow.tracking.fluent: Experiment with name 'Fraud Detection - Random Forest' does not exist. Creating a new experiment.


MLflow experiment configured successfully ✅


In [7]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

In [8]:
import mlflow
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)

mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("Fraud Detection - Random Forest")

# Create Random Forest model
rf_model = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

with mlflow.start_run(run_name="Random Forest Baseline"):

    # Log parameters
    mlflow.log_param("model_type", "RandomForestClassifier")
    mlflow.log_param("n_estimators", 300)
    mlflow.log_param("class_weight", "balanced")
    mlflow.log_param("random_state", 42)
    mlflow.log_param("decision_threshold", 0.30)

    # Train model
    rf_model.fit(X_train_processed, y_train)

    # Probability predictions
    rf_prob = rf_model.predict_proba(X_test_processed)[:, 1]

    # Apply selected threshold
    rf_threshold_pred = (rf_prob >= 0.30).astype(int)

    # Metrics
    precision = precision_score(
        y_test, rf_threshold_pred, zero_division=0
    )

    recall = recall_score(
        y_test, rf_threshold_pred, zero_division=0
    )

    f1 = f1_score(
        y_test, rf_threshold_pred, zero_division=0
    )

    roc_auc = roc_auc_score(
        y_test, rf_prob
    )

    pr_auc = average_precision_score(
        y_test, rf_prob
    )

    # Log metrics
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("f1_score", f1)
    mlflow.log_metric("roc_auc", roc_auc)
    mlflow.log_metric("pr_auc", pr_auc)

    print("MLflow run completed successfully ✅")
    print("Precision:", round(precision, 4))
    print("Recall:", round(recall, 4))
    print("F1 Score:", round(f1, 4))
    print("ROC-AUC:", round(roc_auc, 4))
    print("PR-AUC:", round(pr_auc, 4))

NameError: name 'X_train_processed' is not defined

In [9]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Load dataset
df = pd.read_csv(
    r"C:\Users\lenovo\OneDrive\Desktop\ai-ml project\fraud-detection-mlops\data\raw\credit_card_fraud_10k.csv"
)

# Features and target
X = df.drop(columns=["is_fraud", "transaction_id"])
y = df["is_fraud"]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Feature columns
numerical_features = [
    "amount",
    "transaction_hour",
    "foreign_transaction",
    "location_mismatch",
    "device_trust_score",
    "velocity_last_24h",
    "cardholder_age"
]

categorical_features = [
    "merchant_category"
]

# Preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numerical_features
        ),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ]
)

# Fit preprocessing ONLY on training data
X_train_processed = preprocessor.fit_transform(X_train)

# Transform test data
X_test_processed = preprocessor.transform(X_test)

print("Training data:", X_train_processed.shape)
print("Testing data:", X_test_processed.shape)
print("Required ML variables recreated successfully ✅")

Training data: (8000, 12)
Testing data: (2000, 12)
Required ML variables recreated successfully ✅


In [10]:
import mlflow
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)

# MLflow configuration
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("Fraud Detection - Random Forest")

# Create Random Forest model
rf_model = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

with mlflow.start_run(run_name="Random Forest Baseline"):

    # Log parameters
    mlflow.log_param("model_type", "RandomForestClassifier")
    mlflow.log_param("n_estimators", 300)
    mlflow.log_param("class_weight", "balanced")
    mlflow.log_param("random_state", 42)
    mlflow.log_param("decision_threshold", 0.30)

    # Train model
    rf_model.fit(X_train_processed, y_train)

    # Probability predictions
    rf_prob = rf_model.predict_proba(X_test_processed)[:, 1]

    # Apply CV-selected threshold
    rf_threshold_pred = (rf_prob >= 0.30).astype(int)

    # Calculate metrics
    precision = precision_score(
        y_test, rf_threshold_pred, zero_division=0
    )

    recall = recall_score(
        y_test, rf_threshold_pred, zero_division=0
    )

    f1 = f1_score(
        y_test, rf_threshold_pred, zero_division=0
    )

    roc_auc = roc_auc_score(
        y_test, rf_prob
    )

    pr_auc = average_precision_score(
        y_test, rf_prob
    )

    # Log metrics
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("f1_score", f1)
    mlflow.log_metric("roc_auc", roc_auc)
    mlflow.log_metric("pr_auc", pr_auc)

    print("MLflow run completed successfully ✅")
    print("Precision:", round(precision, 4))
    print("Recall:", round(recall, 4))
    print("F1 Score:", round(f1, 4))
    print("ROC-AUC:", round(roc_auc, 4))
    print("PR-AUC:", round(pr_auc, 4))

MLflow run completed successfully ✅
Precision: 1.0
Recall: 1.0
F1 Score: 1.0
ROC-AUC: 1.0
PR-AUC: 1.0


In [11]:
import mlflow
import mlflow.sklearn

from mlflow.models import infer_signature

# Make sure we're using the same MLflow database
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("Fraud Detection - Random Forest")

# Create input/output examples for model signature
input_example = X_train.iloc[:5]
output_example = rf_model.predict_proba(
    X_train_processed[:5]
)[:, 1]

signature = infer_signature(
    X_train_processed,
    output_example
)

with mlflow.start_run(run_name="Random Forest Model Artifact"):

    # Log model parameters
    mlflow.log_param("model_type", "RandomForestClassifier")
    mlflow.log_param("n_estimators", 300)
    mlflow.log_param("class_weight", "balanced")
    mlflow.log_param("random_state", 42)
    mlflow.log_param("decision_threshold", 0.30)

    # Log metrics
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("f1_score", f1)
    mlflow.log_metric("roc_auc", roc_auc)
    mlflow.log_metric("pr_auc", pr_auc)

    # Log trained model
    mlflow.sklearn.log_model(
        sk_model=rf_model,
        name="fraud_detection_random_forest",
        signature=signature,
        input_example=X_train_processed[:5]
    )

    print("Random Forest model logged to MLflow successfully ✅")

MlflowException: The saved sklearn model references untrusted types. If you are sure loading these types is safe, set the 'skops_trusted_types' parameter when calling 'log_model' or 'save_model' to the list of trusted types. Root error: Untrusted types found in the file: ['sklearn.tree._tree.Tree'].

- sklearn.tree._tree.Tree: sklearn.tree._tree.Tree (the shared node storage for DecisionTree*, RandomForest*, ExtraTrees*, and GradientBoosting* models) stores raw node indices (left_child, right_child, feature) that scikit-learn indexes into without bounds checking. A malicious file can set these to out-of-range values: skops loads the object successfully, but calling .predict() on it can then crash the process (segfault) or read out-of-bounds memory. If you created the file yourself or otherwise fully trust its source, you can load it with trusted=["sklearn.tree._tree.Tree"].

Only add the specific types you have reviewed and trust to the `trusted` argument; avoid passing everything repo

In [12]:
import mlflow
import mlflow.sklearn

from mlflow.models import infer_signature

# Configure MLflow
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("Fraud Detection - Random Forest")

# Use the already trained Random Forest
signature = infer_signature(
    X_train_processed[:5],
    rf_model.predict(X_train_processed[:5])
)

# Start a new MLflow run
with mlflow.start_run(run_name="Random Forest Model - Fixed"):

    model_info = mlflow.sklearn.log_model(
        sk_model=rf_model,
        name="fraud_detection_random_forest",
        serialization_format="cloudpickle",
        signature=signature,
        input_example=X_train_processed[:5]
    )

    mlflow.log_param("decision_threshold", 0.30)

    print("Model saved successfully!")
    print("Model URI:", model_info.model_uri)

2026/09/29 14:01:53 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Model saved successfully!
Model URI: models:/m-3064e946bcaa4f8d8fc37dcd3b3dbe5d


In [13]:
from sklearn.pipeline import Pipeline

# Combine the already-fitted preprocessor and trained Random Forest
fraud_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", rf_model)
    ]
)

print("Production pipeline created successfully ✅")
print(fraud_pipeline)

Production pipeline created successfully ✅
Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['amount', 'transaction_hour',
                                                   'foreign_transaction',
                                                   'location_mismatch',
                                                   'device_trust_score',
                                                   'velocity_last_24h',
                                                   'cardholder_age']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['merchant_category'])])),
                ('model',
                 RandomForestClassifier(class_weight='balanced',
                                        n_estimators=300, n_jobs=-1,
                      

In [14]:
pipeline_prob = fraud_pipeline.predict_proba(X_test)[:, 1]

pipeline_pred = (pipeline_prob >= 0.30).astype(int)

print("Pipeline prediction test successful ✅")
print("Predictions generated:", len(pipeline_pred))
print("First 10 probabilities:", pipeline_prob[:10])

Pipeline prediction test successful ✅
Predictions generated: 2000
First 10 probabilities: [0.         0.00333333 0.         0.         0.         0.00666667
 0.         0.         0.         0.        ]


In [15]:
import mlflow
import mlflow.sklearn

from mlflow.models import infer_signature

# MLflow configuration
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("Fraud Detection - Random Forest")

# Create signature using RAW input
pipeline_signature = infer_signature(
    X_train,
    fraud_pipeline.predict_proba(X_train)[:, 1]
)

with mlflow.start_run(run_name="Production Pipeline"):

    mlflow.log_param("model_type", "RandomForestClassifier")
    mlflow.log_param("n_estimators", 300)
    mlflow.log_param("class_weight", "balanced")
    mlflow.log_param("random_state", 42)
    mlflow.log_param("decision_threshold", 0.30)

    # Log production pipeline
    pipeline_model_info = mlflow.sklearn.log_model(
        sk_model=fraud_pipeline,
        name="fraud_detection_pipeline",
        signature=pipeline_signature,
        input_example=X_train.iloc[:5]
    )

    print("Production pipeline logged to MLflow successfully ✅")
    print("Pipeline Model URI:", pipeline_model_info.model_uri)

c:\Users\lenovo\OneDrive\Desktop\ai-ml project\fraud-detection-mlops\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


MlflowException: The saved sklearn model references untrusted types. If you are sure loading these types is safe, set the 'skops_trusted_types' parameter when calling 'log_model' or 'save_model' to the list of trusted types. Root error: Untrusted types found in the file: ['sklearn.tree._tree.Tree'].

- sklearn.tree._tree.Tree: sklearn.tree._tree.Tree (the shared node storage for DecisionTree*, RandomForest*, ExtraTrees*, and GradientBoosting* models) stores raw node indices (left_child, right_child, feature) that scikit-learn indexes into without bounds checking. A malicious file can set these to out-of-range values: skops loads the object successfully, but calling .predict() on it can then crash the process (segfault) or read out-of-bounds memory. If you created the file yourself or otherwise fully trust its source, you can load it with trusted=["sklearn.tree._tree.Tree"].

Only add the specific types you have reviewed and trust to the `trusted` argument; avoid passing everything repo

In [16]:
import mlflow
import mlflow.sklearn

from mlflow.models import infer_signature

# MLflow configuration
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("Fraud Detection - Random Forest")

# Create signature using raw input
pipeline_signature = infer_signature(
    X_train,
    fraud_pipeline.predict_proba(X_train)[:, 1]
)

with mlflow.start_run(run_name="Production Pipeline - Cloudpickle"):

    mlflow.log_param("model_type", "RandomForestClassifier")
    mlflow.log_param("n_estimators", 300)
    mlflow.log_param("class_weight", "balanced")
    mlflow.log_param("random_state", 42)
    mlflow.log_param("decision_threshold", 0.30)

    pipeline_model_info = mlflow.sklearn.log_model(
        sk_model=fraud_pipeline,
        name="fraud_detection_pipeline",
        serialization_format="cloudpickle",
        signature=pipeline_signature,
        input_example=X_train.iloc[:5]
    )

    print("Production pipeline logged successfully ✅")
    print("Pipeline Model URI:", pipeline_model_info.model_uri)

c:\Users\lenovo\OneDrive\Desktop\ai-ml project\fraud-detection-mlops\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/09/29 14:06:16 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization 

Production pipeline logged successfully ✅
Pipeline Model URI: models:/m-3dee5e2018964fe381d556405f81742d


In [17]:
import mlflow

mlflow.set_tracking_uri("sqlite:///mlflow.db")

with mlflow.start_run(
    run_id="a41cd2999c364438a582cbb10b9657ba"
):
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("f1_score", f1)
    mlflow.log_metric("roc_auc", roc_auc)
    mlflow.log_metric("pr_auc", pr_auc)

print("Production pipeline metrics added successfully ✅")

Production pipeline metrics added successfully ✅
